<a href="https://colab.research.google.com/github/ritvik-123/EMP-Project/blob/master/Module_1_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from pathlib import Path
import os
import re
import json
import time
import joblib
import pandas as pd
import numpy as np

from tqdm.auto import tqdm

from pathlib import Path
from sentence_transformers import SentenceTransformer

import torch

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, multilabel_confusion_matrix, average_precision_score, label_ranking_average_precision_score

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize

from google import genai
from google.genai import types
import os
import getpass


In [ ]:
CSV_PATH = "../Data/"

MODEL_PATH = "../Model/"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

%pwd
%cd /content/drive/MyDrive/EMP/Notebook

In [ ]:

%pwd

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
df = pd.read_csv(CSV_PATH + "Module 1.csv")
print(df.shape)
df.head()

In [ ]:
df = df.drop(columns = ['ID'])

In [ ]:
df.columns.tolist()

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\r", " ").replace("\n"," ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_into_sentences(text):
    text = clean_text(text)
    if not text:
        return []

    sentences = sent_tokenize(text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

sentence_rows = []

for source_id, row in df.reset_index(drop=True).iterrows():
    campus = row.get("Campus", "")
    comment = row.get("Comment", "")
    sentences = split_into_sentences(comment)

    for sentence_index, sentence in enumerate(sentences):
        sentence_rows.append({
            "source_id": source_id,
            "campus": campus,
            "sentence_index": sentence_index,
            "sentence_id": f"{source_id}_{sentence_index}",
            "sentence": sentence
        })
sent_df = pd.DataFrame(sentence_rows)

print(sent_df.shape)
sent_df.head(10)

In [ ]:
sent_df.to_csv(CSV_PATH + "Module 1 sentence.csv", index = False)
print(f"saved sentence-level data to: {CSV_PATH}")

In [ ]:
import getpass
import os
from google import genai

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API key: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

for model in client.models.list():
    print(model.name)
    print("supported actions:", getattr(model, "supported_actions", None))
    print("-" * 80)

In [ ]:
GEMINI_MODEL = "gemini-2.5-flash-lite"

In [ ]:
LABELS = [
    "ideological",
    "institutionalized",
    "interpersonal",
    "internalized"
]

LABEL_DEFINITIONS = """
You are labeling sentences for a qualitative research project.

Classify how much the sentence leans toward each of four forms of oppression.

1. Ideological oppression:
Belief systems, dominant ideas, stereotypes, norms, assumptions, cultural narratives,
or values that justify inequality.

2. Institutionalized oppression:
Oppression embedded in systems, institutions, policies, schools, religion, workplaces,
government, laws, programs, access to resources, or formal structures.

3. Interpersonal oppression:
Oppression expressed through person-to-person interaction, exclusion, judgment,
bias, conflict, mistreatment, microaggressions, or treatment by others.

4. Internalized oppression:
Oppression absorbed into the self, including shame, self-doubt, hiding identity,
discomfort with identity, accepting negative beliefs about oneself, or seeing oneself
as lesser due to dominant narratives.

For each oppression type, return:
- a binary value: 1 if the sentence clearly leans toward that oppression type, otherwise 0
- a score from 0.0 to 1.0 showing how strongly the sentence leans toward that type

Important:
- Do not force a label.
- If the sentence does not clearly lean toward any of the four oppression types, all binary labels should be 0.
- Activity feedback alone should usually receive all 0s.
- General identity reflection without oppression should usually receive all 0s.
- Teaching application should only receive a label if it clearly mentions systems, bias, exclusion, inequity, or oppression.
- Do not create a fifth category.
"""

In [ ]:
def build_prompt(sentence):
    return f"""
{LABEL_DEFINITIONS}

Return only valid JSON in this exact format:

{{
  "ideological": 0,
  "institutionalized": 0,
  "interpersonal": 0,
  "internalized": 0,
  "ideological_score": 0.0,
  "institutionalized_score": 0.0,
  "interpersonal_score": 0.0,
  "internalized_score": 0.0,
  "primary_leaning": "none",
  "rationale": "brief explanation"
}}

Sentence:
\"\"\"{sentence}\"\"\"
"""

def safe_json_parse(text):
    """
    Tries to parse Gemini output as JSON
    Also handles cases where the model wraps JSON in markdown.
    """

    if text is None:
        return None

    text = text.strip()

    # Remove markdown fences if present
    text = re.sub(f"^```json","",text).strip()
    text = re.sub(r"^```", "", text).strip()
    text = re.sub(r"```$","", text).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try extracting the first JSON object
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None

    return None

def label_sentence_with_gemini(sentence, max_retries = 3, sleep_seconds = 2):
    prompt = build_prompt(sentence)

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model = GEMINI_MODEL,
                contents = prompt,
                config=types.GenerateContentConfig(
                    temperature  = 0,
                    response_mime_type="application/json"
                )
            )

            parsed = safe_json_parse(response.text)

            if parsed is None:
                raise ValueError(f"Could not parse JSON: {response.text}")

            # normalize labels
            result = {}
            for label in LABELS:
                # Binary 0/1 label
                result[label] = int(parsed.get(label, 0))

                # Leaning score
                score_col = f"{label}_score"
                result[score_col] = float(parsed.get(score_col, 0.0))

            # If no label is selected, mark none or unclear
            if sum(result[label] for label in LABELS) == 0:
                result["primary_leaning"] = "none"

            else:
                result["primary_leaning"] = max(
                    LABELS,
                    key=lambda label: result[f"{label}_score"]
                )

            result["rationale"] = parsed.get("rationale", "")
            result["gemini_raw"] = response.text

            return result

        except Exception as e:
            if attempt == max_retries - 1:
                result = {}

                for label in LABELS:
                    result[label] = 0
                    result[f"{label}_score"] = 0.0

                result["primary_leaning"] = "none"
                result["rationale"]  = f"ERROR: {str(e)}"
                result["gemini_raw"] = ""

                return result;

            time.sleep(sleep_seconds * (attempt + 1))

In [ ]:
sample_df = sent_df.copy()

In [ ]:
labeled_rows = []

for _,row in tqdm(sample_df.iterrows(), total = len(sample_df)):
    sentence = row["sentence"]
    labels = label_sentence_with_gemini(sentence)

    combined = row.to_dict()
    combined.update(labels)

    # manual review fields
    combined["reviewed"] = 0
    combined["review_notes"] = ""

    labeled_rows.append(combined)

gemini_df = pd.DataFrame(labeled_rows)

gemini_df

In [ ]:
gemini_df.to_csv(CSV_PATH + "Module 1 Sentences Gemini.csv", index = False)
print(f"Saved Gemini weka labels to: {CSV_PATH}")

In [ ]:
reviewed_df = pd.read_csv(CSV_PATH + "Module 1 Sentences Gemini.csv")

print(reviewed_df.shape)
reviewed_df.head()

In [ ]:
reviewed_df = gemini_df.copy()

In [ ]:
TARGET_LABELS = [
    "ideological",
    "institutionalized",
    "interpersonal",
    "internalized"
]

df_model = reviewed_df.copy()

df_model.columns = df_model.columns.str.strip()

# validate required columns
required_cols = ["source_id", "sentence"] + TARGET_LABELS
missing_cols = [col for col in required_cols if col not in df_model.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {', '.join(missing_cols)}")

#Clean text

df_model["sentence"] = df_model["sentence"].fillna("").astype(str).str.strip()
df_model = df_model[df_model["sentence"] != ""].copy()

# clean labels
for label in TARGET_LABELS:
  df_model[label] = pd.to_numeric(df_model[label], errors = "coerce").fillna(0)
  df_model[label] = df_model[label].clip(0,1).astype(int)

# Use source_id as group
df_model["source_id"] = df_model["source_id"].astype(str)

#stage 1 target
df_model["label_count"] = df_model[TARGET_LABELS].sum(axis=1)
df_model["oppression_related"] = (df_model["label_count"]>0).astype(int)

print("Total rows:", len(df_model))
print("\nOppression-related distribution:")
print(df_model["oppression_related"].value_counts())

print("\nFour-label counts:")
print(df_model[TARGET_LABELS].sum())

In [ ]:
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"

embedder =  SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device = device
)

sentences = df_model["sentence"].astype(str).tolist()

X_emb = embedder.encode(
    sentences,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", X_emb.shape)

In [ ]:
X = X_emb
groups = df_model["source_id"].values
y_gate = df_model["oppression_related"].values
Y = df_model[TARGET_LABELS].values

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y_gate, groups=groups)
)

train_df = df_model.iloc[train_idx].copy()
test_df = df_model.iloc[test_idx].copy()

X_train = X[train_idx]
X_test = X[test_idx]

y_gate_train = train_df["oppression_related"].values
y_gate_test = test_df["oppression_related"].values

Y_train = train_df[TARGET_LABELS].values
Y_test = test_df[TARGET_LABELS].values

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain gate distribution:")
print(train_df["oppression_related"].value_counts())

print("\nTest gate distribution:")
print(test_df["oppression_related"].value_counts())

print("\nTrain label counts:")
print(train_df[TARGET_LABELS].sum())

print("\nTest label counts:")
print(test_df[TARGET_LABELS].sum())

In [ ]:
gate_model = LogisticRegression(
    max_iter = 2000,
    class_weight = "balanced",
    solver = "liblinear",
    random_state = 42
)

gate_model.fit(
    X_train,
    y_gate_train,
)

print("stage 1 gate model trained.")

In [ ]:
gate_probs = gate_model.predict_proba(X_test)[:, 1]

for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    gate_preds = (gate_probs >= threshold).astype(int)

    print("=" * 80)
    print("Gate threshold:", threshold)

    print(classification_report(
        y_gate_test,
        gate_preds,
        target_names=["no_strong_leaning", "oppression_related"],
        zero_division=0
    ))

print("Average precision:", average_precision_score(y_gate_test, gate_probs))

In [ ]:
GATE_THRESHOLD = 0.60

In [ ]:
positive_train_mask = train_df["oppression_related"].values == 1

X_type_train = X_train[positive_train_mask]

Y_type_train = train_df.loc[
    positive_train_mask,
    TARGET_LABELS
].values

print("Stage 2 training rows:", len(X_type_train))

print("Stage 2 label counts:")
print(pd.Series(Y_type_train.sum(axis=0), index=TARGET_LABELS))

In [ ]:
type_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear",
        random_state=42
    )
)

type_model.fit(X_type_train, Y_type_train)

print("Stage 2 four-label model trained.")

In [ ]:
positive_test_mask = test_df["oppression_related"].values == 1

X_type_test = X_test[positive_test_mask]

Y_type_test = test_df.loc[
    positive_test_mask,
    TARGET_LABELS
].values

type_probs_positive = type_model.predict_proba(X_type_test)

print("Positive test rows:", len(X_type_test))
print("Positive test label counts:")
print(pd.Series(Y_type_test.sum(axis=0), index=TARGET_LABELS))

In [ ]:
for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    type_preds = (type_probs_positive >= threshold).astype(int)

    # Ensure every positive sentence gets at least one type prediction
    for i in range(len(type_preds)):
        if type_preds[i].sum() == 0:
            type_preds[i, np.argmax(type_probs_positive[i])] = 1

    print("=" * 80)
    print("Type threshold:", threshold)

    print(classification_report(
        Y_type_test,
        type_preds,
        target_names=TARGET_LABELS,
        zero_division=0
    ))

In [ ]:
TYPE_THRESHOLD = 0.55

In [ ]:
def top_k_accuracy(Y_true, probs, k=1):
    top_k_indices = np.argsort(probs, axis=1)[:, -k:]

    hits = []

    for i, indices in enumerate(top_k_indices):
        hits.append(Y_true[i, indices].sum() > 0)

    return float(np.mean(hits))

In [ ]:
if len(Y_type_test) > 0:
    lrap = label_ranking_average_precision_score(
        Y_type_test,
        type_probs_positive
    )

    top1 = top_k_accuracy(Y_type_test, type_probs_positive, k=1)
    top2 = top_k_accuracy(Y_type_test, type_probs_positive, k=2)

    print("Stage 2 LRAP:", lrap)
    print("Stage 2 Top-1 accuracy:", top1)
    print("Stage 2 Top-2 accuracy:", top2)
else:
    print("No positive test rows available for ranking evaluation.")

In [ ]:
def combined_predict(
    gate_probs,
    type_probs,
    gate_threshold,
    type_threshold
):

  preds = np.zeros_like(gate_probs, dtype=int)

  for i in range(len(gate_probs)):
      if gate_probs[i] >= gate_threshold:
          preds[i] = (type_probs[i] >= type_threshold).astype(int)

  return preds


In [ ]:
gate_test_probs = gate_model.predict_proba(X_test)[:, 1]
type_test_probs = type_model.predict_proba(X_test)

# Combined score: "is this oppression-related?" × "which type?"
final_test_scores = type_test_probs * gate_test_probs[:, None]


def combined_predict_from_final_scores(final_scores, threshold):
    return (final_scores >= threshold).astype(int)


for final_threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:
    Y_pred_combined = combined_predict_from_final_scores(
        final_scores=final_test_scores,
        threshold=final_threshold
    )

    print("=" * 80)
    print("FINAL_THRESHOLD:", final_threshold)
    print("Predicted labels:", int(Y_pred_combined.sum()))
    print("True labels:", int(Y_test.sum()))

    print(classification_report(
        Y_test,
        Y_pred_combined,
        target_names=TARGET_LABELS,
        zero_division=0
    ))

In [ ]:
inspect_df = test_df[
    ["source_id", "sentence_id", "sentence", "oppression_related"] + TARGET_LABELS
].copy()

inspect_df["gate_score"] = gate_test_probs
inspect_df["gate_pred"] = (gate_test_probs >= GATE_THRESHOLD).astype(int)

for i, label in enumerate(TARGET_LABELS):
    inspect_df[f"{label}_type_score"] = type_test_probs[:, i]
    inspect_df[f"{label}_final_score"] = final_test_scores[:, i]
    inspect_df[f"pred_{label}"] = Y_pred_combined[:, i]

inspect_df["predicted_primary"] = [
    TARGET_LABELS[int(np.argmax(final_test_scores[i]))]
    if gate_test_probs[i] >= GATE_THRESHOLD
    else "no_strong_leaning"
    for i in range(len(inspect_df))
]

inspect_df.sort_values("gate_score", ascending=False).head(30)